# Data Preprocessing & Feature Engineering

## Objectives
- Load and validate the dataset.
- Separate features from the target variable.
- Assess feature suitability.
- Prepare data for machine learning.

In [94]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [95]:
df = pd.read_csv("../data/AIML Dataset.csv")

In [96]:
df.shape

(6362620, 11)

In [97]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='str')

In [98]:
X = df[["type","amount","oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest",]]

y = df["isFraud"]

In [99]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

X shape: (6362620, 6)
y shape: (6362620,)

Feature columns:
['type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

Target distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [100]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
  X, y, train_size=0.8, test_size=0.2, random_state=42, stratify=y
)

In [101]:
x_train.shape,x_test.shape,y_train.shape,y_test.shape

((5090096, 6), (1272524, 6), (5090096,), (1272524,))

In [102]:
print("Training set:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting set:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True) * 100)

Training set:
isFraud
0    5083526
1       6570
Name: count, dtype: int64
isFraud
0    99.870926
1     0.129074
Name: proportion, dtype: float64

Testing set:
isFraud
0    1270881
1       1643
Name: count, dtype: int64
isFraud
0    99.870887
1     0.129113
Name: proportion, dtype: float64


In [103]:
print(x_train.dtypes)

print("\nTransaction types:")
print(x_train["type"].value_counts())

type                  str
amount            float64
oldbalanceOrg     float64
newbalanceOrig    float64
oldbalanceDest    float64
newbalanceDest    float64
dtype: object

Transaction types:
type
CASH_OUT    1790307
PAYMENT     1721234
CASH_IN     1119026
TRANSFER     426382
DEBIT         33147
Name: count, dtype: int64


In [104]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ["type"]
numeric_features = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

In [105]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
                dtype=np.float32
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numeric_features
        )
    ],
    remainder="drop",
    sparse_threshold=1.0
)

In [106]:
print(preprocessor)

ColumnTransformer(sparse_threshold=1.0,
                  transformers=[('categorical',
                                 OneHotEncoder(dtype=<class 'numpy.float32'>,
                                               handle_unknown='ignore'),
                                 ['type']),
                                ('numerical', 'passthrough',
                                 ['amount', 'oldbalanceOrg', 'newbalanceOrig',
                                  'oldbalanceDest', 'newbalanceDest'])])


In [107]:
print("Training shape:", x_train.shape)
print("Testing shape:", x_test.shape)

print(
    "Training memory usage:",
    round(x_train.memory_usage(deep=True).sum() / (1024 ** 2), 2),
    "MB"
)

print(
    "Testing memory usage:",
    round(x_test.memory_usage(deep=True).sum() / (1024 ** 2), 2),
    "MB"
)

Training shape: (5090096, 6)
Testing shape: (1272524, 6)
Training memory usage: 506.9 MB
Testing memory usage: 126.72 MB


In [108]:
X_train_transformed = preprocessor.fit_transform(x_train)

print("Shape:", X_train_transformed.shape)
print("Data type:", type(X_train_transformed))
print("Data type of values:", X_train_transformed.dtype)

Shape: (5090096, 10)
Data type: <class 'scipy.sparse._csr.csr_matrix'>
Data type of values: float64


In [109]:
train_memory_mb = (
    X_train_transformed.data.nbytes
    + X_train_transformed.indices.nbytes
    + X_train_transformed.indptr.nbytes
) / (1024 ** 2)

print("Transformed training memory:", round(train_memory_mb, 2), "MB")
print("Non-zero values:", X_train_transformed.nnz)
print("Total possible values:", X_train_transformed.shape[0] * X_train_transformed.shape[1])

Transformed training memory: 269.54 MB
Non-zero values: 21856292
Total possible values: 50900960


In [110]:
X_test_transformed = preprocessor.transform(x_test)

print("Test shape:", X_test_transformed.shape)
print("Data type:", type(X_test_transformed))
print("Data type of values:", X_test_transformed.dtype)

Test shape: (1272524, 10)
Data type: <class 'scipy.sparse._csr.csr_matrix'>
Data type of values: float64


In [111]:
print("Training features:", X_train_transformed.shape)
print("Testing features:", X_test_transformed.shape)

print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

print("Training fraud cases:", y_train.sum())
print("Testing fraud cases:", y_test.sum())

Training features: (5090096, 10)
Testing features: (1272524, 10)
Training labels: (5090096,)
Testing labels: (1272524,)
Training fraud cases: 6570
Testing fraud cases: 1643


In [112]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

dummy_model = DummyClassifier(strategy="most_frequent")

dummy_model.fit(X_train_transformed, y_train)

y_pred_dummy = dummy_model.predict(X_test_transformed)

print("Accuracy:", accuracy_score(y_test, y_pred_dummy))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dummy, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dummy))

Accuracy: 0.9987088652159016

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.00      0.00      0.00      1643

    accuracy                           1.00   1272524
   macro avg       0.50      0.50      0.50   1272524
weighted avg       1.00      1.00      1.00   1272524


Confusion Matrix:
[[1270881       0]
 [   1643       0]]
